# SEAL math evaluation

Compare the unsteered model and SEAL on MATH500 and GSM8K using the same engine, prompts and sampling settings. The model is DeepSeek-R1-Distill-Qwen-1.5B. Steering applies `execution - reflection - transition` at layer 20, scale 0.5, on paragraph-break tokens during generation.

Run from this directory after preparing the data in `README.md`. Full answers are saved under `.runtime/`; the notebook displays computed scores and a comparison example. The vectors are the existing experiment artifacts; this evaluation does not retrain them.


In [1]:
import json
import os
from pathlib import Path

import torch
import vllm
from common import load_evaluation_data, make_prompts, score_outputs
from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

from easysteer.steer import StatisticalControlVector
from easysteer.vectors import from_control_vector

MODEL = os.environ.get("EASYSTEER_MODEL", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
DATA_DIR = Path(os.environ.get("EASYSTEER_DATA_DIR", "."))
RESULTS_DIR = Path(".runtime")
LIMIT = int(os.environ.get("EASYSTEER_LIMIT", "0"))  # Zero evaluates each full dataset.
MAX_TOKENS = 8192
RESULTS_DIR.mkdir(exist_ok=True)

llm = LLM(model=MODEL, enable_steer_vector=True, steer_algorithms=["direct"])
tokenizer = llm.get_tokenizer()
sampling = SamplingParams(
    temperature=0, max_tokens=MAX_TOKENS, skip_special_tokens=False
)
print(
    f"vLLM {vllm.__version__}; Torch {torch.__version__}; {torch.cuda.get_device_name(0)}"
)
print(f"Sample limit: {LIMIT or 'full datasets'}; max generated tokens: {MAX_TOKENS}")

vLLM 0.29.0; Torch 2.13.0+cu130; NVIDIA RTX 6000D
Sample limit: full datasets; max generated tokens: 8192


In [2]:
newline_token_ids = sorted(
    token_id
    for token, token_id in tokenizer.get_vocab().items()
    if token.endswith("ĊĊ")
)
parts = {
    name: StatisticalControlVector.import_gguf(f"{name}_avg_vector.gguf")
    for name in ("execution", "reflection", "transition")
}
# Additive directions share a target and selector, so one payload represents their sum.
merged = StatisticalControlVector(
    method="Average",
    directions={
        20: (
            parts["execution"].directions[20]
            - parts["reflection"].directions[20]
            - parts["transition"].directions[20]
        )
    },
    metadata={"composition": "execution - reflection - transition"},
)
steering = SteeringSpec(
    vectors=[
        VectorSpec(
            data=from_control_vector(merged),
            algorithm="direct",
            scale=0.5,
            apply=ApplySpec(generation_tokens=newline_token_ids),
        )
    ]
)

## Accuracy and generation length

The paper's explicit reasoning prefix is retained. GSM8K reference answers are the numeric answers following `####`, rather than the full worked solutions. Generated answers are evaluated with `math_verify`.


In [3]:
results = []
comparisons = {}
for dataset in ("math500", "gsm8k_test"):
    problems, answers = load_evaluation_data(DATA_DIR / f"{dataset}.json", limit=LIMIT)
    prompts = make_prompts(tokenizer, problems)
    records = {}
    for condition, spec in (("baseline", False), ("steered", steering)):
        outputs = llm.generate(prompts, sampling, steering=spec)
        metrics, rows = score_outputs(problems, answers, outputs)
        results.append({"dataset": dataset, "condition": condition, **metrics})
        records[condition] = rows
        (RESULTS_DIR / f"{dataset}_{condition}.json").write_text(
            json.dumps({"metrics": metrics, "rows": rows}, ensure_ascii=False, indent=2)
            + "\n"
        )
        print(
            f"{dataset:12s} {condition:8s}: accuracy={metrics['accuracy']:.2%}, "
            f"mean tokens={metrics['mean_generated_tokens']:.1f}, n={metrics['questions']}"
        )
        del outputs
    improved = [
        i
        for i, (base, steer) in enumerate(
            zip(records["baseline"], records["steered"], strict=True)
        )
        if not base["correct"] and steer["correct"]
    ]
    example_index = improved[0] if improved else 0
    comparisons[dataset] = {
        condition: rows[example_index] for condition, rows in records.items()
    }

print("\nSummary")
for row in results:
    print(
        f"{row['dataset']:12s} {row['condition']:8s}  "
        f"{row['accuracy']:7.2%}  {row['mean_generated_tokens']:8.1f} tokens"
    )

math500      baseline: accuracy=63.80%, mean tokens=4488.8, n=500


math500      steered : accuracy=68.60%, mean tokens=3633.6, n=500


gsm8k_test   baseline: accuracy=75.06%, mean tokens=2749.5, n=1319


gsm8k_test   steered : accuracy=80.74%, mean tokens=1688.5, n=1319

Summary
math500      baseline   63.80%    4488.8 tokens
math500      steered    68.60%    3633.6 tokens
gsm8k_test   baseline   75.06%    2749.5 tokens
gsm8k_test   steered    80.74%    1688.5 tokens


In [4]:
for dataset, pair in comparisons.items():
    print(f"\n{dataset}: {pair['baseline']['problem']}")
    print(f"Reference: {pair['baseline']['answer']}")
    for condition, row in pair.items():
        print(f"{condition} (correct={row['correct']}), final 600 characters:")
        print(row["output"][-600:])


math500: If $f(x) = \frac{3x-2}{x-2}$, what is the value of $f(-2) +f(-1)+f(0)$? Express your answer as a common fraction.
Reference: \frac{14}{3}
baseline (correct=False), final 600 characters:
 f(-2) + f(-1) + f(0) is equal to:

(3*(-2) - 2)/( (-2) - 2 ) + (3*(-1) - 2)/( (-1) - 2 ) + (3*0 - 2)/(0 - 2 )

Which is:

(-6 - 2)/(-4) + (-3 - 2)/(-3) + (-2)/(-2 )

Which is:

(-8)/(-4) + (-5)/(-3) + (-2)/(-2 )

Which is:

2 + 5/3 + 1

Which is 2 + 1 + 5/3 = 3 + 5/3 = 14/3.

So, same result.

So, 14/3 is correct.

Wait, but let me think again. Maybe I can compute f(-2) + f(-1) + f(0) as a single expression.

Wait, f(x) = (3x - 2)/(x - 2). So, f(-2) + f(-1) + f(0) is equal to:

(3*(-2) - 2)/( (-2) - 2 ) + (3*(-1) - 2)/( (-1) - 2 ) + (3*0 - 2)/(0 - 2 )

Which is:

(-6 - 2)/(-4) + (-3 - 2)/(-3)
steered (correct=True), final 600 characters:
} = 2
   \]

2. Calculate \( f(-1) \):
   \[
   f(-1) = \frac{3(-1) - 2}{-1 - 2} = \frac{-3 - 2}{-3} = \frac{-5}{-3} = \frac{5}{3}
   \]

3. Calculate \( f(0